# LSTM for NER on Conll2003

We read the sentences and the labels from github, we use `keras.preprocessing.text.Tokenizer` for encoding tokens, we pad sequences to a fixed length and then we train an LSTM model for doing Named Entity Recognition.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nluninja/text-mining-dataviz-aa2526/blob/main/05-LSTM/NLP05-03_Entity_Extraction/NLP05-03_LSTM_4_NER-v2.ipynb)

In [ ]:
import os, sys

# --- Colab Setup ---
# If running on Colab, clone the repo and set up the working directory
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    if not os.path.exists('text-mining-dataviz-aa2526'):
        !git clone https://github.com/nluninja/text-mining-dataviz-aa2526.git
    os.chdir('text-mining-dataviz-aa2526/05-LSTM/NLP05-03_Entity_Extraction')
    !pip install -q seqeval

In [1]:
import urllib
import sklearn
import logging
import os
import numpy as np
from seqeval.metrics import classification_report
from utils import dataio, kerasutils, modelutils
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical, plot_model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

## Read Dataset

In [2]:
data_dir = os.path.join('data', 'conll03')
raw_train, ner_train, output_labels = dataio.load_conll_data('train.txt', dir_path=data_dir, only_tokens=True)
raw_valid, ner_valid, _ = dataio.load_conll_data('valid.txt', dir_path=data_dir, only_tokens=True)
raw_test, ner_test, _ = dataio.load_conll_data('test.txt', dir_path=data_dir, only_tokens=True)

Reading file data\conll03\train.txt
Read 14027 sentences
Reading file data\conll03\valid.txt
Read 3249 sentences
Reading file data\conll03\test.txt
Read 3452 sentences


In [3]:
print(raw_train[0])
print(ner_train[0])

['German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
['B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O']


---

## Feature Engineering

### Token Ordinal Encoding

In [4]:
# integer encode sequences of words
token_tokenizer = Tokenizer()    # Automatically lowers tokens
token_tokenizer.fit_on_texts(raw_train + raw_valid + raw_test)
train_sequences = token_tokenizer.texts_to_sequences(raw_train)
test_sequences = token_tokenizer.texts_to_sequences(raw_test)
valid_sequences = token_tokenizer.texts_to_sequences(raw_valid)

tag2idx = { tag: idx for idx, tag in enumerate(output_labels) }
idx2tag = { idx: tag for tag, idx in tag2idx.items() }
ner_train_sequences = [[tag2idx[tag] for tag in sentence] for sentence in ner_train]
ner_test_sequences  = [[tag2idx[tag] for tag in sentence] for sentence in ner_test ]
ner_valid_sequences = [[tag2idx[tag] for tag in sentence] for sentence in ner_valid]

In [5]:
print(raw_test[5])
print(test_sequences[5])
for i in test_sequences[5]:
    print(f'{i} : {token_tokenizer.index_word[i]}')

['China', 'controlled', 'most', 'of', 'the', 'match', 'and', 'saw', 'several', 'chances', 'missed', 'until', 'the', '78th', 'minute', 'when', 'Uzbek', 'striker', 'Igor', 'Shkvyrin', 'took', 'advantage', 'of', 'a', 'misdirected', 'defensive', 'header', 'to', 'lob', 'the', 'ball', 'over', 'the', 'advancing', 'Chinese', 'keeper', 'and', 'into', 'an', 'empty', 'net', '.']
[175, 3041, 236, 4, 1, 117, 10, 1196, 455, 2862, 2908, 429, 1, 5406, 314, 81, 24523, 897, 5270, 13682, 269, 2081, 4, 7, 24524, 3034, 2958, 6, 11361, 1, 1003, 72, 1, 4769, 905, 5066, 10, 107, 36, 4925, 310, 3]
175 : china
3041 : controlled
236 : most
4 : of
1 : the
117 : match
10 : and
1196 : saw
455 : several
2862 : chances
2908 : missed
429 : until
1 : the
5406 : 78th
314 : minute
81 : when
24523 : uzbek
897 : striker
5270 : igor
13682 : shkvyrin
269 : took
2081 : advantage
4 : of
7 : a
24524 : misdirected
3034 : defensive
2958 : header
6 : to
11361 : lob
1 : the
1003 : ball
72 : over
1 : the
4769 : advancing
905 : chine

In [6]:
vocabulary_size = len(token_tokenizer.word_counts)
print(vocabulary_size)

26861


In [7]:
print(raw_train[0])
print(ner_train_sequences[0])
for i in ner_train_sequences[0]:
    print(f'{i} : {idx2tag[i]}')

['German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
[0, 8, 8, 8, 0, 8, 8]
0 : B-MISC
8 : O
8 : O
8 : O
0 : B-MISC
8 : O
8 : O


### Sequence Padding

In [8]:
sequence_len = np.array([len(s) for s in train_sequences])
longest_sequence = sequence_len.max()
print(f'Longest sequence: {longest_sequence}')

print([(str(p) + '%', np.percentile(sequence_len, p)) for p in range(75,101, 5)])

Longest sequence: 113
[('75%', 22.0), ('80%', 26.0), ('85%', 29.0), ('90%', 32.0), ('95%', 37.69999999999891), ('100%', 113.0)]


In [9]:
max_sequence_len = 50
X_train = pad_sequences(train_sequences, maxlen=max_sequence_len, padding='post', truncating='post')
X_test = pad_sequences(test_sequences, maxlen=max_sequence_len, padding='post', truncating='post')
X_valid = pad_sequences(valid_sequences, maxlen=max_sequence_len, padding='post', truncating='post')

Y_train = pad_sequences(ner_train_sequences, maxlen=max_sequence_len, value=tag2idx['O'], padding='post', truncating='post')
Y_test = pad_sequences(ner_test_sequences, maxlen=max_sequence_len, value=tag2idx['O'], padding='post', truncating='post')
Y_valid = pad_sequences(ner_valid_sequences, maxlen=max_sequence_len, value=tag2idx['O'], padding='post', truncating='post')

Y_train = to_categorical(Y_train, num_classes=len(output_labels), dtype='int32')
Y_test = to_categorical(Y_test, num_classes=len(output_labels), dtype='int32')
Y_valid = to_categorical(Y_valid, num_classes=len(output_labels), dtype='int32')

In [10]:
print(X_train[0])
print(Y_train[0])

[ 207  709    6 3628  228 7656    3    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0]
[[1 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 1]
 [1 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 1]
 [0 0 0 0 

In [11]:
token_tokenizer.index_word[0] = '_PAD_'

In [12]:
X_train = np.array(X_train)
Y_train = np.array(Y_train)
X_test = np.array(X_test)
Y_test = np.array(Y_test)
X_valid = np.array(X_valid)
Y_valid = np.array(Y_valid)

In [13]:
print(X_train.shape)
print(Y_train.shape)

(14027, 50)
(14027, 50, 9)


## Build, train and evaluate an LSTM model

In [ ]:
glove_matrix=None
if USE_GLOVE:
    glove_embedding_path = os.path.join('embeddings', 'glove.6B.100d.txt')
    embedding_dim = 100

    # Download GloVe embeddings if not present
    if not os.path.isfile(glove_embedding_path):
        import urllib.request, zipfile
        print("Downloading GloVe embeddings...")
        glove_url = "https://nlp.stanford.edu/data/glove.6B.zip"
        zip_path = os.path.join('embeddings', 'glove.6B.zip')
        os.makedirs('embeddings', exist_ok=True)
        urllib.request.urlretrieve(glove_url, zip_path)
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extract('glove.6B.100d.txt', 'embeddings')
        os.remove(zip_path)
        print("Done.")

    glove_matrix = kerasutils.load_glove_embedding_matrix(glove_embedding_path, token_tokenizer.word_index, embedding_dim)

we use a BiLSTM architecture.

![Bi-LSTM](schema.png "Bi-LSTM")

model = kerasutils.create_paper_BiLSTM(vocabulary_size+1, max_sequence_len, len(output_labels),
                                 use_glove=USE_GLOVE, glove_matrix=glove_matrix)

best_model_file = os.path.join('models', 'lstm-conll-best-model.h5')
os.makedirs('models', exist_ok=True)

checkpoint = ModelCheckpoint(
    best_model_file,
    save_weights_only=True,
    save_best_only=True
)

# More aggressive Early Stopping
# early_stopping_callback = EarlyStopping(monitor="val_loss", min_delta=0.01, patience=1, verbose=1, mode="auto", restore_best_weights=False)

#Less aggressive Early Stopping
early_stopping_callback = EarlyStopping(monitor="val_loss", min_delta=0.01, patience=3, verbose=1, mode="auto", restore_best_weights=True)


![Bi-LSTM](model.png "Bi-LSTM")


![Bi-LSTM](.\model.png "Bi-LSTM")

In [17]:
%%time
batch_size = 10
history = model.fit(X_train, 
          Y_train, 
          batch_size=batch_size, 
          epochs=1,
          verbose=2,
          callbacks=[checkpoint, early_stopping_callback],
          validation_data=(X_valid, Y_valid)
         )

1403/1403 - 235s - loss: 0.3009 - accuracy: 0.9159 - precision: 0.9416 - recall: 0.8900 - val_loss: 0.1352 - val_accuracy: 0.9619 - val_precision: 0.9701 - val_recall: 0.9543 - 235s/epoch - 167ms/step
CPU times: total: 17min 12s
Wall time: 3min 55s


In [18]:
kerasutils.print_model_memory_usage(batch_size, model)

Model size: 13.26 MB


In [19]:
print(f'Model latency in predictions: {modelutils.compute_prediction_latency(X_test, model):.3} s')

108/108 [==============================] - 9s 49ms/step
Model latency in predictions: 0.00413 s


In [20]:
datasets = [('Training Set', X_train, Y_train), ('Test Set', X_test, Y_test), ('Validation Set', X_valid, Y_valid)]

for title, X, Y in datasets:
    Y_pred = model.predict(X, batch_size=batch_size)
    Y_pred = np.array(np.argmax(Y_pred, axis=-1))
    Y = np.array(np.argmax(Y, axis=-1))
    Y, Y_pred = kerasutils.remove_seq_padding1(X, Y, Y_pred)
    let_y_true, let_y_pred = modelutils.from_encode_to_literal_labels(Y, Y_pred, idx2tag)
    
    print(title)
    print(classification_report(let_y_true, let_y_pred, digits=3))
    print('\n')

1403/1403 [==============================] - 39s 28ms/step
Training Set
              precision    recall  f1-score   support

         LOC      0.873     0.896     0.885      7134
        MISC      0.777     0.616     0.687      3435
         ORG      0.752     0.723     0.737      6312
         PER      0.915     0.927     0.921      6589

   micro avg      0.842     0.817     0.829     23470
   macro avg      0.829     0.791     0.807     23470
weighted avg      0.838     0.817     0.826     23470



346/346 [==============================] - 10s 29ms/step
Test Set
              precision    recall  f1-score   support

         LOC      0.807     0.864     0.835      1655
        MISC      0.698     0.596     0.643       700
         ORG      0.729     0.646     0.685      1657
         PER      0.892     0.878     0.885      1579

   micro avg      0.798     0.770     0.784      5591
   macro avg      0.782     0.746     0.762      5591
weighted avg      0.794     0.770     0.781  

In [23]:
i = 5
sentence = X_test[i]
y_pred = model.predict(X_test)
y_pred = np.argmax(y_pred, axis=-1)
y_pred = y_pred[i]
y_true = np.argmax(Y_test, axis=-1)[i]
for idx in range(len(sentence)):
    print(f'{token_tokenizer.index_word[sentence[idx]]:15}  {idx2tag[y_true[idx]]:6} | {idx2tag[y_pred[idx]]}')


108/108 [==============================] - 5s 46ms/step
china            B-LOC  | B-LOC
controlled       O      | O
most             O      | O
of               O      | O
the              O      | O
match            O      | O
and              O      | O
saw              O      | O
several          O      | O
chances          O      | O
missed           O      | O
until            O      | O
the              O      | O
78th             O      | O
minute           O      | O
when             O      | O
uzbek            B-MISC | B-MISC
striker          O      | O
igor             B-PER  | B-PER
shkvyrin         I-PER  | I-PER
took             O      | O
advantage        O      | O
of               O      | O
a                O      | O
misdirected      O      | O
defensive        O      | O
header           O      | O
to               O      | O
lob              O      | O
the              O      | O
ball             O      | O
over             O      | O
the              O      | O
adv